# Rule Extraction — Tree-Based Classification Rules (Stage 4–5)

Trains dedicated **shallow** rule models (Random Forest + Decision Tree) on the
best feature-selection set from the benchmark, walks every root→leaf path into IF-THEN
rules, scores them, and cross-checks the two models.

**Prerequisite:** Run `02_Benchmark.ipynb` (the benchmark) first.

**Output per dataset** (`outputs/{dataset}/rules/`)
```
rules/
  rf/        rules.json · rules.csv · rules_with_probes.csv · rules_human.txt ·
             rules_summary.json · genes_in_rules.csv
  dt/        rules.json · rules.csv · rules_with_probes.csv · rules_human.txt ·
             rules_summary.json · genes_in_rules.csv
  compare/   crosscheck.json   (rules / literals confirmed by BOTH models)
```

`rules.csv` antecedents are gene-level (e.g. `ERBB2>8.12`); `rules_with_probes.csv`
spells out the exact probe ID(s) backing each condition instead (useful when
several probes collapse to one gene and you need to know which one fired).
`genes_in_rules.csv` is the distinct gene list actually used by the *surviving*
rules — not the full SHAP top-K — with backing probe IDs, rule counts, and
which classes each gene appears in.

**Config:** `configs/interpretation.yaml → interpretation.rules`

**Metrics** — *data-based*: support, confidence, error, lift, class_specificity ·
*prediction-based*: **fidelity** (agreement with the source model, used by the
rule-guided ANN in stage 10) · *set-level*: coverage, mean_fidelity.

## Setup

In [1]:
import sys
import json
import warnings
from pathlib import Path

# Make src.* importable from the notebooks/ directory
REPO_ROOT = Path("..").resolve()
sys.path.insert(0, str(REPO_ROOT))

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning, module="sklearn")

import numpy as np
import pandas as pd

from src.pipeline import GeneExpressionPipeline

pipe = GeneExpressionPipeline(
    config_root=str(REPO_ROOT / "configs"),
    output_root=str(REPO_ROOT / "outputs"),
    use_wandb=False,
)

## Run Rule Extraction

Picks the best FS set (by benchmark RF), then trains the shallow rule models
configured in `interpretation.rules.models` (default `rf` + `decisiontree`) and
extracts rules from each.

`pipe.run_rule_extraction(['GEO-Breast-20711'])` — specific datasets  
`pipe.run_rule_extraction()` — all enabled datasets

In [2]:
results = pipe.run_rule_extraction(dataset_names=None)
results

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ Rule Extraction — Tree-Based Classification Rules                                                               │
│ Stage 4-5: dedicated RF/DecisionTree rule models per dataset                                                    │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

──────────────────────────────────────────── Dataset: GEO-Breast-20711 ────────────────────────────────────────────

    Best FS feature set: mrmr_miq (by balanced_accuracy)

  ✓ Reused prebuilt mapping: 45,118 probe→gene entries from GSE20711_mapping.csv

  → Rule extraction (rf)

...

  ✓ SHAP complete → C:\Users\Bao Nguyen\LearningWithClaude\LV_code\outputs\CuMiDa-Brain-50161\rules\dt\shap

  ✓ Cross-check rf vs decisiontree: 1 identical rule(s), 12 shared gene-direction literal(s) (literal Jaccard=0.04)
-> C:\Users\Bao Nguyen\LearningWithClaude\LV_code\outputs\CuMiDa-Brain-50161\rules\compare

        CuMiDa-Brain-50161 — extracted rules (FS: boruta)         
┏━━━━━━━━━━━━━━┳━━━━━━━━━┳━━━━━━━┳━━━━━━━┳━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃        model ┃ n_rules ┃ cls:0 ┃ cls:1 ┃ cls:2 ┃ cls:3 ┃ cls:4 ┃
┡━━━━━━━━━━━━━━╇━━━━━━━━━╇━━━━━━━╇━━━━━━━╇━━━━━━━╇━━━━━━━╇━━━━━━━┩
│           rf │     100 │    20 │    20 │    20 │    20 │    20 │
│ decisiontree │       5 │     1 │     1 │     1 │     1 │     1 │
└──────────────┴─────────┴───────┴───────┴───────┴───────┴───────┘

    Cross-check: 1 identical rule(s), 12 shared gene-direction literal(s) (literal Jaccard=0.04)

{'GEO-Breast-20711': {'fs_method': 'mrmr_miq',
  'models': {'rf': {'n_rules': 80,
    'per_class': {'Basal': 20, 'HER2': 20, 'LumA': 20, 'LumB': 20}},
   'decisiontree': {'n_rules': 4,
    'per_class': {'Basal': 1, 'HER2': 1, 'LumA': 1, 'LumB': 1}}},
  'compare': {'n_a': 80,
   'n_b': 4,
   'n_overlap': 1,
   'jaccard': 0.0128,
   'overlap': [{'literals': ['ERBB2=High', 'MMRN1=Low'],
     'consequent': 'Class=HER2',
     'rf_confidence': 1.0,
     'decisiontree_confidence': 1.0}],
   'literal_overlap': 13,
   'literal_jaccard': 0.1066,
   'shared_literals': [{'gene': 'ERBB2',
     'direction': 'High',
     'consequent': 'Class=HER2'},
    {'gene': 'ERBB2', 'direction': 'Low', 'consequent': 'Class=Basal'},
    {'gene': 'ERBB2', 'direction': 'Low', 'consequent': 'Class=LumA'},
    {'gene': 'ERBB2', 'direction': 'Low', 'consequent': 'Class=LumB'},
    {'gene': 'KCNK15', 'direction': 'High', 'consequent': 'Class=LumB'},
    {'gene': 'KCNK15', 'direction': 'Low', 'consequent': 'Class=Basal'

## Gene Descriptions — Biological Annotation (separate step)

Enrich the genes appearing in the rules with biological info from each dataset's
platform file (`Platform_Annotations/…`): **Gene title, Entrez ID, GenBank,
chromosome, GO Biological Process / Cellular Component / Molecular Function**.
Writes `gene_description.csv` + `gene_description.md` next to each
`genes_in_rules.csv` under `outputs/{dataset}/rules/{rf,dt}/`.

This is a **separate step**: it reads existing rule outputs (no re-training), so
it can run any time after rule extraction.

In [ ]:
# Separate step — annotate rule genes from the platform files.
gene_desc = pipe.describe_genes_in_rules(dataset_names=None)
gene_desc

## Inspect Outputs

In [3]:
DATASET = 'GEO-Breast-20711'
rules_root = pipe.output_root / DATASET / 'rules'
print('Rules root:', rules_root)
for sub in sorted(p.name for p in rules_root.iterdir() if p.is_dir()):
    print(' ', sub, '→', [f.name for f in sorted((rules_root / sub).iterdir())])

Rules root: C:\Users\Bao Nguyen\LearningWithClaude\LV_code\outputs\GEO-Breast-20711\rules
  compare → ['crosscheck.json']
  dt → ['rules.csv', 'rules.json', 'rules_human.txt', 'rules_summary.json']
  rf → ['rules.csv', 'rules.json', 'rules_human.txt', 'rules_summary.json']


### Random Forest rules (ensemble — many candidate rules)

In [4]:
rf_rules = pd.read_csv(rules_root / 'rf' / 'rules.csv')
rf_summary = json.loads((rules_root / 'rf' / 'rules_summary.json').read_text())
print('RF set-level:', {k: rf_summary[k] for k in ('n_rules', 'coverage', 'mean_fidelity')})
rf_rules.head(10)

RF set-level: {'n_rules': 80, 'coverage': 0.9778, 'mean_fidelity': 0.9567}


,antecedents,consequents,support,confidence,error,lift,class_specificity,fidelity,strength_score,n_conditions
0,"242960_at<=4.079, ERBB2<=11.17, MYB>9.031, KNL...",Class=LumA,0.1333,1.0000,0.0000,6.9231,1.0000,1.0000,0.923077,5
1,"241536_at<=5.086, CDK12<=8.472, KNL1<=5.627, E...",Class=LumA,0.1444,0.9231,0.0769,6.3905,0.8462,1.0000,0.852071,4
2,"233392_at<=3.523, CCNB2<=6.637, SLC40A1>5.814",Class=LumA,0.1444,0.9231,0.0769,6.3905,0.8462,0.9231,0.852071,3
3,"241536_at<=5.086, MYB>9.018, SVEP1>2.637, TPX2...",Class=LumA,0.1444,0.9231,0.0769,6.3905,0.8462,0.9231,0.852071,5
4,"MYH11<=6.028, CCNB2>6.221, ERBB2>5.481",Class=HER2,0.2444,1.0000,0.0000,3.4615,1.0000,1.0000,0.846154,3
5,"SVEP1<=3.288, ERBB2>10.78",Class=HER2,0.2444,1.0000,0.0000,3.4615,1.0000,1.0000,0.846154,2
6,"CDK1>6.359, ERBB2>4.344",Class=HER2,0.2444,1.0000,0.0000,3.4615,1.0000,1.0000,0.846154,2
7,"LOC401433<=3.916, ERBB2>5.573",Class=HER2,0.2444,1.0000,0.0000,3.4615,1.0000,1.0000,0.846154,2
8,"TNFRSF10B<=4.676, CCNB2>6.637, ERBB2>5.552",Class=HER2,0.2444,1.0000,0.0000,3.4615,1.0000,1.0000,0.846154,3
9,"RNF186<=4.339, CCNB2<=6.637, USP43>5.194",Class=LumA,0.1222,1.0000,0.0000,6.9231,1.0000,1.0000,0.846154,3


### Decision Tree rules (single tree — few, exact, easy to read)

In [5]:
dt_rules = pd.read_csv(rules_root / 'dt' / 'rules.csv')
print((rules_root / 'dt' / 'rules_human.txt').read_text())
dt_rules

IF 242904_x_at<=4.787 AND ERBB2<=4.28 AND KNL1<=5.87 AND USP43>5.017 THEN Class=LumA (conf=0.93, support=0.16, lift=6.43, fidelity=1.00)
IF 242904_x_at<=4.787 AND ERBB2>4.28 AND KCNK15>2.387 THEN Class=HER2 (conf=1.00, support=0.21, lift=3.46, fidelity=1.00)
IF 242904_x_at<=4.787 AND ERBB2<=4.28 AND KNL1>5.87 AND KCNK15>4.021 THEN Class=LumB (conf=1.00, support=0.17, lift=4.09, fidelity=1.00)
IF 242904_x_at<=4.787 AND ERBB2>4.28 AND KCNK15<=2.387 THEN Class=HER2 (conf=1.00, support=0.03, lift=3.46, fidelity=1.00)


,antecedents,consequents,support,confidence,error,lift,class_specificity,fidelity,strength_score,n_conditions
0,"242904_x_at<=4.787, ERBB2<=4.28, KNL1<=5.87, U...",Class=LumA,0.1556,0.9286,0.0714,6.4286,0.8571,1.0,0.928571,4
1,"242904_x_at<=4.787, ERBB2>4.28, KCNK15>2.387",Class=HER2,0.2111,1.0000,0.0000,3.4615,1.0000,1.0,0.730769,3
2,"242904_x_at<=4.787, ERBB2<=4.28, KNL1>5.87, KC...",Class=LumB,0.1667,1.0000,0.0000,4.0909,1.0000,1.0,0.681818,4
3,"242904_x_at<=4.787, ERBB2>4.28, KCNK15<=2.387",Class=HER2,0.0333,1.0000,0.0000,3.4615,1.0000,1.0,0.115385,3


### Rule count per class (RF)

In [6]:
rf_rules.groupby('consequents').agg(
    n_rules=('consequents', 'count'),
    mean_confidence=('confidence', 'mean'),
    mean_fidelity=('fidelity', 'mean'),
    mean_support=('support', 'mean'),
    mean_lift=('lift', 'mean'),
).round(4)

,n_rules,mean_confidence,mean_fidelity,mean_support,mean_lift
consequents,,,,,
Class=Basal,20,0.8868,0.9472,0.2706,2.9560
Class=HER2,20,1.0000,1.0000,0.2444,3.4615
Class=LumA,20,0.9367,0.9608,0.1344,6.4847
Class=LumB,20,0.9205,0.9259,0.1972,3.7655


## Cross-Check: RF vs Decision Tree

Rules confirmed by **both** a single tree and the ensemble are the
highest-trust set. Two levels are reported:
- **identical rules** — same (gene, direction) literal set + class (strict)
- **shared literals** — single `GENE=High/Low → Class` facts both models agree on
  (lenient; informative even when full antecedents differ).

In [7]:
cc = json.loads((rules_root / 'compare' / 'crosscheck.json').read_text())
print(f"{cc['model_a']} ({cc['n_a']} rules) vs {cc['model_b']} ({cc['n_b']} rules)")
print(f"Identical rules : {cc['n_overlap']} (Jaccard={cc['jaccard']})")
print(f"Shared literals : {cc['literal_overlap']} (Jaccard={cc['literal_jaccard']})")
print()
pd.DataFrame(cc['shared_literals'])

rf (80 rules) vs decisiontree (4 rules)
Identical rules : 0 (Jaccard=0.0)
Shared literals : 8 (Jaccard=0.0645)



,gene,direction,consequent
0,242904_x_at,Low,Class=LumA
1,ERBB2,High,Class=HER2
2,ERBB2,Low,Class=LumA
3,ERBB2,Low,Class=LumB
4,KCNK15,High,Class=LumB
5,KNL1,High,Class=LumB
6,KNL1,Low,Class=LumA
7,USP43,High,Class=LumA


## Faithfulness Check

Verify a rule's confidence against the raw data using the exact per-probe
conditions stored in `antecedent_raw`. Expected: recomputed ≈ reported.

In [ ]:
from src.helper import load_dataset, load_fs_reduced_matrix

ds_cfg = pipe.config_loader.get_dataset_config(DATASET)
fs_method = results[DATASET]['fs_method']
data = load_dataset(dataset_path=Path(ds_cfg['path']), dataset_type=ds_cfg.get('type', 'auto'))
y = data['y']

# Load the FS feature matrix the rule models were trained on.
if fs_method == 'raw':
    X_sel, feat = data['X'], list(data['feature_names'])
else:
    csv_dir = pipe.output_root / 'feature_selection' / DATASET / fs_method / 'processed_datasets'
    X_sel, feat = load_fs_reduced_matrix(csv_dir)

cards = json.loads((rules_root / 'rf' / 'rules.json').read_text())
r = cards['rules'][0]
print('Rule:', r['text'])
print('Reported:', r['metrics'])

mask = np.ones(len(y), dtype=bool)
for cond in r['antecedent_raw']:
    idx = feat.index(cond['probe'])
    mask &= (X_sel[:, idx] <= cond['threshold']) if cond['op'] == '<=' else (X_sel[:, idx] > cond['threshold'])

cls = r['consequent']['class_idx']
n_ant = int(mask.sum())
emp_conf = (y[mask] == cls).mean() if n_ant else 0
print(f"\nAntecedent fires on {n_ant} samples; empirical confidence = {emp_conf:.4f}")
print(f"Match reported confidence: {abs(emp_conf - r['metrics']['confidence']) < 1e-3}")

## Export Summary

In [9]:
rows = []
for ds_name, info in results.items():
    for model_name, summ in info.get('models', {}).items():
        rows.append({
            'dataset': ds_name,
            'fs_method': info['fs_method'],
            'model': model_name,
            'n_rules': summ['n_rules'],
            'identical_overlap': (info.get('compare') or {}).get('n_overlap'),
            'literal_overlap': (info.get('compare') or {}).get('literal_overlap'),
        })
pd.DataFrame(rows)

,dataset,fs_method,model,n_rules,identical_overlap,literal_overlap
0,GEO-Breast-20711,mrmr_miq,rf,80,0,8
1,GEO-Breast-20711,mrmr_miq,decisiontree,4,0,8
